# ❤️ Heart Disease Prediction
### Random Forest + XGBoost Ensemble

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install xgboost -q

In [ ]:
import zipfile, os, glob
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# Unzip
zip_path = '/content/drive/MyDrive/ml_datasets/heart-disease-uci.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/heart')

csv_files = glob.glob('/content/heart/**/*.csv', recursive=True)
print('Found:', csv_files)
df = pd.read_csv(csv_files[0])
print(df.head())
print(df.shape)

In [ ]:
# Target column
target_col = 'target' if 'target' in df.columns else df.columns[-1]
print('Target:', target_col)

X = df.drop(target_col, axis=1).fillna(df.median(numeric_only=True))
y = (df[target_col] > 0).astype(int)

print('Class distribution:\n', y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

In [ ]:
rf  = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
xgb = XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=42, eval_metric='logloss')

model = VotingClassifier(estimators=[('rf', rf), ('xgb', xgb)], voting='soft')
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
print(f'Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}%')
print(classification_report(y_test, y_pred))

In [ ]:
save_dir = '/content/drive/MyDrive/ml_models'
os.makedirs(save_dir, exist_ok=True)

with open(f'{save_dir}/heart_model.pkl', 'wb') as f:
    pickle.dump(model, f)
with open(f'{save_dir}/heart_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print('✅ heart_model.pkl saved!')
print('✅ heart_scaler.pkl saved!')